In [0]:
# Silver Layer: Clean and transform Bronze tables to Silver tables
from pyspark.sql.functions import col,lit,concat

bronze_schema = "salesproject.bronze_layer"
silver_schema = "salesproject.silver_layer"

entities = [
    ("customers", "customer_id"),
    ("transactions", "transaction_id"),
    ("accounts", "account_id"),
    ("opportunities", "opportunity_id"),
    ("leads", "lead_id")
]

for entity, pk in entities:
    bronze_table = f"{bronze_schema}.{entity}_raw"
    silver_table = f"{silver_schema}.{entity}_cleaned"
    
    # Read raw data from Bronze
    df = spark.table(bronze_table)
    
    # Basic cleaning: drop duplicates, filter null PKs
    df_clean = df.dropDuplicates([pk]).filter(col(pk).isNotNull())
    
    # Write cleaned data to Silver
    df_clean.write.format("delta").mode("overwrite").saveAsTable(silver_table)
    print(f"Transformed {bronze_table} to {silver_table}")
    
# Optionally, display a sample from one Silver table
display(spark.table(f"{silver_schema}.customers_cleaned").limit(5))

In [0]:
# Silver Layer: Create enriched tables via joins
# Join customers_cleaned with transactions_cleaned
customers = spark.table(f"{silver_schema}.customers_cleaned")
transactions = spark.table(f"{silver_schema}.transactions_cleaned")

customer_transactions = transactions.join(customers, on="customer_id", how="inner") \
    .select(
        transactions["transaction_id"],
        transactions["customer_id"],
        concat(customers["first_name"], lit(" "), customers["last_name"]).alias("customer_name"),
        transactions["transaction_ts"],
        transactions["amount"].alias("transaction_amount"),
        transactions["currency"],
        transactions["channel"],
        transactions["status"]
    )
customer_transactions.write.format("delta").mode("overwrite").saveAsTable(f"{silver_schema}.customer_transactions")
print(f"Created {silver_schema}.customer_transactions")

default_display_count = 5
display(customer_transactions.limit(default_display_count))

# Join accounts_cleaned with opportunities_cleaned
accounts = spark.table(f"{silver_schema}.accounts_cleaned")
opportunities = spark.table(f"{silver_schema}.opportunities_cleaned")

account_opportunities = opportunities.join(accounts, on="account_id", how="inner")
account_opportunities.write.format("delta").mode("overwrite").saveAsTable(f"{silver_schema}.account_opportunities")
print(f"Created {silver_schema}.account_opportunities")

display(account_opportunities.limit(default_display_count))